# Example: BBBC014 — Two Cell Lines Dose-Response

This notebook demonstrates morphological profiling of two different cell lines (A549 and MCF7) treated with various compounds at multiple concentrations.

**Dataset Information:**
- **Source:** [Broad Institute BBBC014](https://bbbc.broadinstitute.org/BBBC014)
- **Description:** Multi-channel images of human cell lines treated with compounds.
- **Metadata:** Provided in a `metadata.csv` file mapping filenames to experimental conditions.

**Workflow Overview:**
1. Load metadata and calculate log concentrations.
2. Link multi-channel images (2 channels) with metadata.
3. Extract DINOv2 embeddings and compute intensity/morphological properties.
4. Visualize cell line differences and dose-response effects.
5. Interactively explore the multi-channel phenospace.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model
import pandas as pd

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

## 1. Paths & Configuration

In [ ]:
DATA_DIR = "/datasets/BBBC/BBBC014/BBBC014_v1_images"
METADATA_PATH = "/datasets/BBBC/BBBC014/metadata.csv"

## 2. Load Metadata & Preprocessing

We load the metadata and derive the cell line and log concentration values.

In [ ]:
df_metadata = pd.read_csv(METADATA_PATH)

# Map cell types to line names
df_metadata["cell_line"] = df_metadata["cell_type"].apply(
    lambda x: "MCF7" if x in ["A", "B", "C", "D"] else "A549"
)

# Map concentration points to log concentrations
map_concentration = {
    1: -7.0, 2: -7.523, 3: -8.0, 4: -8.523, 5: -9.0, 6: -9.523, 
    7: -10.0, 8: -10.523, 9: -11.0, 10: -11.523, 11: -12.0, 12: -13.0
}
df_metadata["log_concentration"] = df_metadata["concentration_point"].map(map_concentration)

df_metadata.head()

## 3. Multi-Channel Data Discovery

BBBC014 uses two channels per image. We specify both filename columns to ensure PhenoMe loads them as multi-channel inputs.

In [ ]:
from phenome.utils import make_dataframe_metadata_fn

file_df = pheno.find_files(
    str(DATA_DIR),
    metadata_fn=make_dataframe_metadata_fn(
        df_metadata, 
        filename_column=["filename_channel_1", "filename_channel_2"]
    ),
)
file_df.head()

## 4. Feature Extraction

Extract embeddings and compute morphological properties. Note that we use a large `resize_size` and `pad_size` as these are full fields-of-view.

In [ ]:
pheno.process_images(
    wrapper,
    batch_size=8,
    channel_mode="split",
    resize_size=1358,
    pad_size=1360,
)

df_properties = pheno.compute_properties(
    property_preset="full_extended",
    n_jobs=4,
)

## 5. Visualization

Visualize the difference between cell lines using PCA or t-SNE.

In [ ]:
pheno.plot_tsne(
    color_by="cell_line",
    n_components=2,
)

In [ ]:
pheno.plot_umap(
    color_by="log_concentration",
    filters={"cell_line": "A549"},
    n_components=2,
)

## 6. Interactive Exploration

Inspect the multi-channel images and their positions in the phenospace.

In [ ]:
e = pheno.create_interactive_explorer()